In [1]:
import random
import time

import tensorflow as tf
from tensorflow.keras import Sequential

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix,  precision_score, recall_score, f1_score, accuracy_score,roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Normalizer, OrdinalEncoder, MinMaxScaler,  LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import SelectKBest, chi2, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

import numpy as np
import pandas as pd


In [2]:
data_1 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Monday-WorkingHours.pcap_ISCX.csv")
data_2 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Tuesday-WorkingHours.pcap_ISCX.csv")
data_3 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Wednesday-workingHours.pcap_ISCX.csv")
data_4 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
data_5 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
data_6 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Friday-WorkingHours-Morning.pcap_ISCX.csv")
data_7 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
data_8 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

In [3]:
data_list = [data_1, data_2, data_3, data_4, data_5, data_6, data_7, data_8]
df = pd.concat(data_list)
rows, cols = df.shape

print('New dimension:')
print(f'Number of rows: {rows}')
print(f'Number of columns: {cols}')
print(f'Total cells: {rows * cols}')

New dimension:
Number of rows: 2830743
Number of columns: 79
Total cells: 223628697


In [4]:
for d in data_list: del d

# **Hyperparameter optimization for 1d cnn-based network intrusion detection using ga and pso**

In [ ]:
df[' Label'] = df[' Label'].map(lambda a: 0 if a == 'BENIGN' else 1)
df = df.drop([' Bwd PSH Flags', ' Bwd URG Flags', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'], axis=1)
df.duplicated().sum()


np.int64(308381)

In [ ]:
df =  df.drop_duplicates(keep="first")
df.isnull().sum().sort_values(ascending = False)


,0
Flow Bytes/s,353
Flow Duration,0
Destination Port,0
Total Backward Packets,0
Total Length of Fwd Packets,0
...,...
Idle Mean,0
Idle Std,0
Idle Max,0
Idle Min,0


In [ ]:
df.dropna(inplace=True)
df = df[~df.isin([np.nan, np.inf, -np.inf]).any(axis=1)]

In [ ]:
df.shape

(2520798, 71)

In [ ]:
df=df.groupby(' Label').filter(lambda x:len(x)>10000)
df[' Label'].value_counts()

,count
Label,
0,2095057
1,425741


In [ ]:
X = df.drop(columns=[' Label'])
y = df[' Label']

In [ ]:
# Split the data into a training set (70%), and a temporary set (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Split the temporary set into a validation set (20% of the total data) and a test set (20% of the total data)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.125, random_state=42)

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
input_shape = (X_train.shape[1], 1)
output_classes = 1

In [ ]:
model = Sequential()

model.add(Conv1D(filters=64, kernel_size=9, activation='relu', input_shape=(input_shape)))
model.add(MaxPooling1D(pool_size=5))

model.add(Conv1D(filters=128, kernel_size=9, activation='relu'))
model.add(MaxPooling1D(pool_size=2)) # corrected value

model.add(Dropout(0.113585))
model.add(Flatten())

for _ in range(2):
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.113585))

model.add(Dense(output_classes, activation='sigmoid'))

optimizer = Adam(learning_rate= 0.001193)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 62, 64)         │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 4, 128)         │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 206,337 (806.00 KB)

 Trainable params: 206,337 (806.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train, y_train, batch_size=64, epochs=99, validation_data=(X_val, y_val))


Epoch 1/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 89s 3ms/step - accuracy: 0.9757 - loss: 0.0544 - val_accuracy: 0.9806 - val_loss: 0.0428
Epoch 2/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 0.9802 - loss: 0.0429 - val_accuracy: 0.9816 - val_loss: 0.0389
Epoch 3/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 0.9808 - loss: 0.0410 - val_accuracy: 0.9817 - val_loss: 0.0369
Epoch 4/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 0.9813 - loss: 0.0409 - val_accuracy: 0.9825 - val_loss: 0.0361
Epoch 5/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 66s 2ms/step - accuracy: 0.9816 - loss: 0.0391 - val_accuracy: 0.9815 - val_loss: 0.0350
Epoch 6/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 0.9822 - loss: 0.0372 - val_accuracy: 0.9830 - val_loss: 0.0349
Epoch 7/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 0.9826 - loss: 0.0366 - val_accuracy: 0.9838 - val_loss: 0.0326
Epoch 8/99
27572/27572 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 

In [ ]:
y_pred = model.predict(np.expand_dims(X_test, axis=2))


15755/15755 ━━━━━━━━━━━━━━━━━━━━ 22s 1ms/step


In [ ]:
y_pred = (y_pred > 0.5)

cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test, y_pred)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")


Confusion matrix
----------------
[[415925   3327]
 [  3956  80952]]

Attack-class metrics
--------------------
Accuracy : 0.9856 (98.56%)
Detection rate : 0.9534 (95.34%)
Precision      : 0.9605 (96.05%)
FPR            : 0.0079 (0.79%)
F1 score       : 0.9570 (95.70%)


In [ ]:
model.save("/content/drive/MyDrive/Models_Table_7/1DCNN_Kilichev_CIC-IDS2017.keras")

# **DCNN: a novel binary and multi‑class network intrusion detection model via deep convolutional neural network**

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
df.columns = (
    df.columns
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

print("Number of columns:", len(df.columns))
print(df.columns.tolist())

Number of columns: 79
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag C

In [ ]:
duplicated_column_names = df.columns[
    df.columns.duplicated()
].tolist()

print("Duplicated column names:", duplicated_column_names)

if len(duplicated_column_names) > 0:
    raise ValueError(
        "Duplicate column names were detected and must be resolved."
    )

Duplicated column names: []


In [ ]:
label_candidates = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
]

print("Label candidates:", label_candidates)

if len(label_candidates) != 1:
    raise ValueError(
        "Exactly one label column was expected."
    )

label_column = label_candidates[0]

print("Selected label column:", label_column)

Label candidates: ['Label']
Selected label column: Label


In [ ]:
label_column = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
][0]

df[label_column] = (
    df[label_column]
    .astype(str)
    .str.strip()
    .str.replace("�", "-", regex=False)
)

print(df[label_column].value_counts(dropna=False))

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [ ]:
label_column = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
][0]

df["binary_target"] = np.where(
    df[label_column].str.upper().eq("BENIGN"),
    0,
    1
).astype(np.int8)

binary_counts_before_cleaning = (
    df["binary_target"]
    .value_counts()
    .sort_index()
)

binary_proportions_before_cleaning = (
    df["binary_target"]
    .value_counts(normalize=True)
    .sort_index()
)

print("Counts before cleaning:")
print(binary_counts_before_cleaning)

print("\nProportions before cleaning:")
print(binary_proportions_before_cleaning)

Counts before cleaning:
binary_target
0    2273097
1     557646
Name: count, dtype: int64

Proportions before cleaning:
binary_target
0    0.803004
1    0.196996
Name: proportion, dtype: float64


In [ ]:
duplicate_check_columns = [
    column
    for column in df.columns
    if column != "source_file"
]

duplicate_mask = df.duplicated(
    subset=duplicate_check_columns,
    keep="first"
)

print("Duplicate rows found:", int(duplicate_mask.sum()))
print(
    "Duplicate percentage:",
    float(duplicate_mask.mean() * 100)
)

Duplicate rows found: 308381
Duplicate percentage: 10.893994968812075


In [ ]:
duplicate_check_columns = [
    column
    for column in df.columns
    if column != "source_file"
]

data = (
    df
    .drop_duplicates(
        subset=duplicate_check_columns,
        keep="first"
    )
    .reset_index(drop=True)
)

print("Shape after duplicate removal:", data.shape)

Shape after duplicate removal: (2522362, 80)


In [ ]:
excluded_identifier_names = {
    "flow id",
    "flowid",
    "source ip",
    "src ip",
    "destination ip",
    "dst ip",
    "timestamp",
    "source_file"
}

excluded_identifier_columns = [
    column
    for column in data.columns
    if column.lower() in excluded_identifier_names
]

print("Identifier columns found:")
print(excluded_identifier_columns)

Identifier columns found:
[]


In [ ]:
label_column = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
][0]

columns_not_used_as_predictors = (
    excluded_identifier_columns
    + [label_column, "binary_target"]
)

X_raw = df.drop(
    columns=columns_not_used_as_predictors,
    errors="ignore"
).copy()

y_raw = df["binary_target"].copy()

print("Raw predictor shape:", X_raw.shape)
print("Target shape:", y_raw.shape)
print("Removed columns:", columns_not_used_as_predictors)

Raw predictor shape: (2830743, 78)
Target shape: (2830743,)
Removed columns: ['Label', 'binary_target']


In [ ]:
for column in X_raw.columns:
    X_raw[column] = pd.to_numeric(
        X_raw[column],
        errors="coerce"
    )

X_raw = X_raw.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Numeric predictor shape:", X_raw.shape)
print(
    "Total missing or non-finite values:",
    int(X_raw.isna().sum().sum())
)

Numeric predictor shape: (2830743, 78)
Total missing or non-finite values: 5734


In [ ]:
invalid_value_summary = (
    X_raw
    .isna()
    .sum()
    .sort_values(ascending=False)
)

invalid_value_summary = invalid_value_summary[
    invalid_value_summary > 0
]

invalid_value_summary

,0
Flow Packets/s,2867
Flow Bytes/s,2867


In [ ]:
entirely_missing_columns = X_raw.columns[
    X_raw.isna().all()
].tolist()

X_raw = X_raw.drop(
    columns=entirely_missing_columns
)

constant_columns = [
    column
    for column in X_raw.columns
    if X_raw[column].nunique(dropna=True) <= 1
]

X_raw = X_raw.drop(
    columns=constant_columns
)

print("Entirely missing columns removed:")
print(entirely_missing_columns)

print("\nConstant columns removed:")
print(constant_columns)

print("\nRemaining predictors:", X_raw.shape[1])

Entirely missing columns removed:
[]

Constant columns removed:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Remaining predictors: 70


In [ ]:
complete_row_mask = X_raw.notna().all(axis=1)

removed_incomplete_rows = int(
    (~complete_row_mask).sum()
)

X_clean = (
    X_raw
    .loc[complete_row_mask]
    .reset_index(drop=True)
)

y_clean = (
    y_raw
    .loc[complete_row_mask]
    .reset_index(drop=True)
)

print("Rows removed:", removed_incomplete_rows)
print("Clean predictor shape:", X_clean.shape)
print("Clean target shape:", y_clean.shape)

Rows removed: 2867
Clean predictor shape: (2827876, 70)
Clean target shape: (2827876,)


In [ ]:
print(
    "Remaining missing values:",
    int(X_clean.isna().sum().sum())
)

print(
    "All predictors numeric:",
    bool(
        all(
            np.issubdtype(dtype, np.number)
            for dtype in X_clean.dtypes
        )
    )
)

print(
    "Predictor and target lengths equal:",
    len(X_clean) == len(y_clean)
)

print(
    "Target values:",
    sorted(y_clean.unique().tolist())
)

Remaining missing values: 0
All predictors numeric: True
Predictor and target lengths equal: True
Target values: [0, 1]


In [ ]:
clean_class_summary = pd.DataFrame({
    "count": y_clean.value_counts().sort_index(),
    "proportion": y_clean.value_counts(
        normalize=True
    ).sort_index()
})

clean_class_summary.index = [
    "BENIGN",
    "ATTACK"
]

clean_class_summary

,count,proportion
BENIGN,2271320,0.803189
ATTACK,556556,0.196811


In [ ]:
X_train_frame, X_test_frame, y_train_series, y_test_series = (
    train_test_split(
        X_clean,
        y_clean,
        test_size=0.25,
        random_state=42,
        shuffle=True,
        stratify=y_clean
    )
)

print("Training predictors:", X_train_frame.shape)
print("Testing predictors:", X_test_frame.shape)
print("Training targets:", y_train_series.shape)
print("Testing targets:", y_test_series.shape)

print("\nTraining target proportions:")
print(
    y_train_series
    .value_counts(normalize=True)
    .sort_index()
)

print("\nTesting target proportions:")
print(
    y_test_series
    .value_counts(normalize=True)
    .sort_index()
)

Training predictors: (2120907, 70)
Testing predictors: (706969, 70)
Training targets: (2120907,)
Testing targets: (706969,)

Training target proportions:
binary_target
0    0.803189
1    0.196811
Name: proportion, dtype: float64

Testing target proportions:
binary_target
0    0.803189
1    0.196811
Name: proportion, dtype: float64


In [ ]:
standard_scaler = StandardScaler(
    with_mean=True,
    with_std=True
)

X_train_scaled = standard_scaler.fit_transform(
    X_train_frame
)

X_test_scaled = standard_scaler.transform(
    X_test_frame
)

X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)

y_train_array = y_train_series.to_numpy(
    dtype=np.float32
)

y_test_array = y_test_series.to_numpy(
    dtype=np.float32
)

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Training shape: (2120907, 70)
Testing shape: (706969, 70)


In [ ]:
training_feature_means = X_train_scaled.mean(axis=0)
training_feature_standard_deviations = X_train_scaled.std(axis=0)

print(
    "Mean absolute training mean:",
    float(np.mean(np.abs(training_feature_means)))
)

print(
    "Mean training standard deviation:",
    float(
        np.mean(training_feature_standard_deviations)
    )
)

print(
    "Smallest training standard deviation:",
    float(
        np.min(training_feature_standard_deviations)
    )
)

print(
    "Largest training standard deviation:",
    float(
        np.max(training_feature_standard_deviations)
    )
)

Mean absolute training mean: 5.702704530818892e-09
Mean training standard deviation: 0.9999998807907104
Smallest training standard deviation: 0.9999992251396179
Largest training standard deviation: 1.000000238418579


In [ ]:
X_train_cnn = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_cnn = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

print("CNN training shape:", X_train_cnn.shape)
print("CNN testing shape:", X_test_cnn.shape)
print("Sequence length:", X_train_cnn.shape[1])
print("Input channels:", X_train_cnn.shape[2])

CNN training shape: (2120907, 70, 1)
CNN testing shape: (706969, 70, 1)
Sequence length: 70
Input channels: 1


In [ ]:
input_sequence_length = X_train_cnn.shape[1]

length_after_conv_1 = (
    input_sequence_length - 3
) // 1 + 1

length_after_pooling = (
    length_after_conv_1 - 3
) // 1 + 1

length_after_conv_2 = (
    length_after_pooling - 3
) // 1 + 1

length_after_upsampling = (
    length_after_conv_2 * 3
)

length_after_conv_3 = (
    length_after_upsampling - 3
) // 1 + 1

flattened_dimension = (
    length_after_conv_3 * 256
)

dimension_summary = pd.DataFrame({
    "stage": [
        "Input",
        "Conv1D 256",
        "MaxPooling1D",
        "Conv1D 128",
        "UpSampling1D",
        "Conv1D 256",
        "Flatten"
    ],
    "sequence_length": [
        input_sequence_length,
        length_after_conv_1,
        length_after_pooling,
        length_after_conv_2,
        length_after_upsampling,
        length_after_conv_3,
        flattened_dimension
    ]
})

dimension_summary

,stage,sequence_length
0,Input,70
1,Conv1D 256,68
2,MaxPooling1D,66
3,Conv1D 128,64
4,UpSampling1D,192
5,Conv1D 256,190
6,Flatten,48640


In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

number_of_input_features = X_train_cnn.shape[1]

dcnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(number_of_input_features, 1)
    ),

    tf.keras.layers.Conv1D(
        filters=256,
        kernel_size=3,
        strides=1,
        padding="valid",
        activation="relu"
    ),

    tf.keras.layers.MaxPooling1D(
        pool_size=3,
        strides=1,
        padding="valid"
    ),

    tf.keras.layers.Conv1D(
        filters=128,
        kernel_size=3,
        strides=1,
        padding="valid",
        activation="relu"
    ),

    tf.keras.layers.UpSampling1D(
        size=3
    ),

    tf.keras.layers.Conv1D(
        filters=256,
        kernel_size=3,
        strides=1,
        padding="valid",
        activation="relu"
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        units=256,
        activation="relu"
    ),

    tf.keras.layers.Dropout(
        rate=0.1
    ),

    tf.keras.layers.Dense(
        units=128,
        activation="relu"
    ),

    tf.keras.layers.Dropout(
        rate=0.1
    ),

    tf.keras.layers.Dense(
        units=128,
        activation="relu"
    ),

    tf.keras.layers.Dropout(
        rate=0.1
    ),

    tf.keras.layers.Dense(
        units=1,
        activation="sigmoid"
    )
])

dcnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 68, 256)        │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 66, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 128)        │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling1d (UpSampling1D)    │ (None, 192, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 190, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 48640)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    12,452,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,699,649 (48.45 MB)

 Trainable params: 12,699,649 (48.45 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
dcnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),
        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5
        ),
        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5
        ),
        tf.keras.metrics.AUC(
            name="roc_auc",
            curve="ROC"
        ),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

In [ ]:
training_history = dcnn_model.fit(
    X_train_cnn,
    y_train_array,
    validation_split=0.10,
    epochs=30,
    batch_size=128,
    shuffle=True,
    verbose=1
)

Epoch 1/30
14913/14913 ━━━━━━━━━━━━━━━━━━━━ 99s 5ms/step - accuracy: 0.9768 - loss: 0.0535 - pr_auc: 0.9887 - precision: 0.9373 - recall: 0.9457 - roc_auc: 0.9967 - val_accuracy: 0.9849 - val_loss: 0.0352 - val_pr_auc: 0.9941 - val_precision: 0.9535 - val_recall: 0.9700 - val_roc_auc: 0.9985
Epoch 2/30
14913/14913 ━━━━━━━━━━━━━━━━━━━━ 60s 4ms/step - accuracy: 0.9854 - loss: 0.0343 - pr_auc: 0.9945 - precision: 0.9545 - recall: 0.9723 - roc_auc: 0.9985 - val_accuracy: 0.9876 - val_loss: 0.0299 - val_pr_auc: 0.9959 - val_precision: 0.9584 - val_recall: 0.9792 - val_roc_auc: 0.9990
Epoch 3/30
14913/14913 ━━━━━━━━━━━━━━━━━━━━ 60s 4ms/step - accuracy: 0.9868 - loss: 0.0311 - pr_auc: 0.9955 - precision: 0.9592 - recall: 0.9747 - roc_auc: 0.9988 - val_accuracy: 0.9885 - val_loss: 0.0288 - val_pr_auc: 0.9960 - val_precision: 0.9604 - val_recall: 0.9815 - val_roc_auc: 0.9990
Epoch 4/30
14913/14913 ━━━━━━━━━━━━━━━━━━━━ 60s 4ms/step - accuracy: 0.9876 - loss: 0.0291 - pr_auc: 0.9961 - precision: 

In [ ]:
test_probabilities = dcnn_model.predict(
    X_test_cnn,
    batch_size=128,
    verbose=1
).reshape(-1)

prediction_threshold = 0.5

y_pred = (
    test_probabilities >= prediction_threshold
).astype(np.int8)

print("Probability minimum:", test_probabilities.min())
print("Probability maximum:", test_probabilities.max())
print("Predicted benign:", int((y_pred == 0).sum()))
print("Predicted attacks:", int((y_pred == 1).sum()))

5524/5524 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step
Probability minimum: 0.0
Probability maximum: 1.0
Predicted benign: 567736
Predicted attacks: 139233


In [ ]:
cm = confusion_matrix(y_test_array, y_pred)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_array, y_pred)

recall = recall_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[566949    881]
 [   787 138352]]

Attack-class metrics
--------------------
Accuracy : 0.9976 (99.76%)
Detection rate : 0.9943 (99.43%)
Precision      : 0.9937 (99.37%)
FPR            : 0.0016 (0.16%)
F1 score       : 0.9940 (99.40%)


In [ ]:
dcnn_model.save("/content/drive/MyDrive/Models_Table_7/1DCCN_buhl.keras")

# **Enhanced Hybrid Deep Learning Models-Based Anomaly Detection Method for Two-Stage Binary and Multi-Class Classification of Attacks in Intrusion Detection Systems**

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
df.columns = (
    df.columns
    .astype(str)
    .str.replace(
        "\ufeff",
        "",
        regex=False
    )
    .str.strip()
)

print("Column count:", len(df.columns))
print(df.columns.tolist())


duplicated_column_names = df.columns[
    df.columns.duplicated(
        keep=False
    )
].tolist()

print("Duplicated column names:")
print(duplicated_column_names)

df = df.loc[
    :,
    ~df.columns.duplicated(
        keep="first"
    )
].copy()

print(
    "Shape after duplicate-column removal:",
    df.shape
)



label_candidates = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
]

print("Label candidates:", label_candidates)

if len(label_candidates) != 1:
    raise ValueError(
        "Exactly one label column was expected."
    )

label_column = label_candidates[0]

df[label_column] = (
    df[label_column]
    .astype(str)
    .str.strip()
)

print(df[label_column].value_counts(dropna=False))

Column count: 79
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count'

In [ ]:
label_column = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
][0]

df["binary_target"] = np.where(
    df[label_column]
    .str.upper()
    .eq("BENIGN"),
    0,
    1
).astype(np.int8)

binary_summary = pd.DataFrame({
    "count": (
        df["binary_target"]
        .value_counts()
        .sort_index()
    ),
    "proportion": (
        df["binary_target"]
        .value_counts(normalize=True)
        .sort_index()
    )
})

binary_summary.index = [
    "BENIGN",
    "ATTACK"
]

binary_summary

,count,proportion
BENIGN,2273097,0.803004
ATTACK,557646,0.196996


In [ ]:
duplicate_check_columns = [
    column
    for column in df.columns
    if column != "source_file"
]

duplicate_mask = df.duplicated(
    subset=duplicate_check_columns,
    keep="first"
)

print(
    "Exact duplicate rows:",
    int(duplicate_mask.sum())
)

df = (
    df.loc[~duplicate_mask]
    .reset_index(drop=True)
)

print(
    "Shape after duplicate removal:",
    df.shape
)

Exact duplicate rows: 308381
Shape after duplicate removal: (2522362, 80)


In [ ]:
label_column = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
][0]

X_raw = df.drop(
    columns=[
        label_column,
        "binary_target",
        "source_file"
    ],
    errors="ignore"
).copy()

y_raw = df["binary_target"].copy()

print("Raw predictor shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

Raw predictor shape: (2522362, 78)
Target shape: (2522362,)


In [ ]:
for column in X_raw.columns:
    X_raw[column] = pd.to_numeric(
        X_raw[column],
        errors="coerce"
    )

X_raw = X_raw.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "Total missing or infinite values:",
    int(X_raw.isna().sum().sum())
)

Total missing or infinite values: 3128


In [ ]:
constant_columns = [
    column
    for column in X_raw.columns
    if X_raw[column].nunique(
        dropna=True
    ) <= 1
]

print("Constant columns:")
print(constant_columns)

X_raw = X_raw.drop(
    columns=constant_columns
)

print(
    "Remaining predictor count:",
    X_raw.shape[1]
)

entirely_missing_columns = [
    column
    for column in X_raw.columns
    if X_raw[column].isna().all()
]

print("Entirely missing columns:")
print(entirely_missing_columns)

X_raw = X_raw.drop(
    columns=entirely_missing_columns
)

print(
    "Remaining predictor count:",
    X_raw.shape[1]
)



Constant columns:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Remaining predictor count: 70
Entirely missing columns:
[]
Remaining predictor count: 70


In [ ]:
column_means = X_raw.mean(axis=0)

X_imputed = X_raw.fillna(
    column_means
)

print(
    "Missing values after imputation:",
    int(X_imputed.isna().sum().sum())
)

print(
    "Predictor count after cleaning:",
    X_imputed.shape[1]
)

Missing values after imputation: 0
Predictor count after cleaning: 70


In [ ]:
paper_feature_count = 69
loaded_feature_count = X_imputed.shape[1]

print("Paper feature count:", paper_feature_count)
print("Loaded feature count:", loaded_feature_count)
print(
    "Difference:",
    loaded_feature_count - paper_feature_count
)

Paper feature count: 69
Loaded feature count: 70
Difference: 1


In [ ]:
benign_indices = y_raw[
    y_raw == 0
].index.to_numpy()

attack_indices = y_raw[
    y_raw == 1
].index.to_numpy()

print("Available benign rows:", len(benign_indices))
print("Available attack rows:", len(attack_indices))

if len(benign_indices) < 99000:
    raise ValueError(
        "There are fewer than 99,000 benign records."
    )

if len(attack_indices) < 76368:
    raise ValueError(
        "There are fewer than 76,368 attack records."
    )

random_generator = np.random.default_rng(42)

sampled_benign_indices = (
    random_generator.choice(
        benign_indices,
        size=99000,
        replace=False
    )
)

sampled_attack_indices = (
    random_generator.choice(
        attack_indices,
        size=76368,
        replace=False
    )
)

sampled_indices = np.concatenate([
    sampled_benign_indices,
    sampled_attack_indices
])

random_generator.shuffle(
    sampled_indices
)

X_sampled = (
    X_imputed
    .loc[sampled_indices]
    .reset_index(drop=True)
)

y_sampled = (
    y_raw
    .loc[sampled_indices]
    .reset_index(drop=True)
)

print("Sampled predictor shape:", X_sampled.shape)
print(y_sampled.value_counts().sort_index())

Available benign rows: 2096484
Available attack rows: 425878
Sampled predictor shape: (175368, 70)
binary_target
0    99000
1    76368
Name: count, dtype: int64


In [ ]:
from sklearn.neighbors import LocalOutlierFactor
local_outlier_factor = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=False,
    n_jobs=-1
)

lof_predictions = (
    local_outlier_factor.fit_predict(
        X_sampled
    )
)

inlier_mask = lof_predictions == 1

X_inliers = (
    X_sampled
    .loc[inlier_mask]
    .reset_index(drop=True)
)

y_inliers = (
    y_sampled
    .loc[inlier_mask]
    .reset_index(drop=True)
)

print(
    "Outliers removed:",
    int((~inlier_mask).sum())
)

print(
    "Remaining rows:",
    len(X_inliers)
)

print("\nPost-LOF class counts:")
print(
    y_inliers
    .value_counts()
    .sort_index()
)

Outliers removed: 8769
Remaining rows: 166599

Post-LOF class counts:
binary_target
0    93568
1    73031
Name: count, dtype: int64


In [ ]:
X_train_frame, X_test_frame, y_train, y_test = (
    train_test_split(
        X_inliers,
        y_inliers,
        test_size=0.15,
        random_state=42,
        shuffle=True,
        stratify=y_inliers
    )
)

print("Training predictor shape:", X_train_frame.shape)
print("Test predictor shape:", X_test_frame.shape)

print("\nTraining class counts:")
print(
    y_train
    .value_counts()
    .sort_index()
)

print("\nTest class counts:")
print(
    y_test
    .value_counts()
    .sort_index()
)

Training predictor shape: (141609, 70)
Test predictor shape: (24990, 70)

Training class counts:
binary_target
0    79533
1    62076
Name: count, dtype: int64

Test class counts:
binary_target
0    14035
1    10955
Name: count, dtype: int64


In [ ]:
minmax_scaler = MinMaxScaler(
    feature_range=(0, 1)
)

X_train_scaled = minmax_scaler.fit_transform(
    X_train_frame
)

X_test_scaled = minmax_scaler.transform(
    X_test_frame
)

X_train_scaled = X_train_scaled.astype(
    np.float32
)

X_test_scaled = X_test_scaled.astype(
    np.float32
)

y_train_array = y_train.to_numpy(
    dtype=np.int8
)

y_test_array = y_test.to_numpy(
    dtype=np.int8
)

print(
    "Training minimum:",
    float(X_train_scaled.min())
)

print(
    "Training maximum:",
    float(X_train_scaled.max())
)

Training minimum: 0.0
Training maximum: 1.0


In [ ]:
from imblearn.over_sampling import ADASYN
print("Training distribution before ADASYN:")
print(
    pd.Series(y_train_array)
    .value_counts()
    .sort_index()
)

adasyn = ADASYN(
    sampling_strategy="auto",
    random_state=42,
    n_neighbors=5
)

X_train_adasyn, y_train_adasyn = (
    adasyn.fit_resample(
        X_train_scaled,
        y_train_array
    )
)

print("\nDistribution after ADASYN:")
print(
    pd.Series(y_train_adasyn)
    .value_counts()
    .sort_index()
)

Training distribution before ADASYN:
0    79533
1    62076
Name: count, dtype: int64

Distribution after ADASYN:
0    79533
1    79871
Name: count, dtype: int64


In [ ]:
from imblearn.over_sampling import SMOTE
adasyn_class_counts = (
    pd.Series(y_train_adasyn)
    .value_counts()
)

majority_class_label = int(
    adasyn_class_counts.idxmax()
)

minority_class_label = int(
    adasyn_class_counts.idxmin()
)

majority_class_count = int(
    adasyn_class_counts.max()
)

minority_class_count = int(
    adasyn_class_counts.min()
)

print(
    "Majority count after ADASYN:",
    majority_class_count
)

print(
    "Minority count after ADASYN:",
    minority_class_count
)

if minority_class_count < majority_class_count:
    smote = SMOTE(
        sampling_strategy={
            minority_class_label:
            majority_class_count
        },
        random_state=42,
        k_neighbors=5
    )

    X_train_smote, y_train_smote = (
        smote.fit_resample(
            X_train_adasyn,
            y_train_adasyn
        )
    )
else:
    X_train_smote = X_train_adasyn.copy()
    y_train_smote = y_train_adasyn.copy()

print("\nDistribution after SMOTE:")
print(
    pd.Series(y_train_smote)
    .value_counts()
    .sort_index()
)

Majority count after ADASYN: 79871
Minority count after ADASYN: 79533

Distribution after SMOTE:
0    79871
1    79871
Name: count, dtype: int64


In [ ]:
from imblearn.under_sampling import EditedNearestNeighbours
edited_nearest_neighbours = (
    EditedNearestNeighbours(
        sampling_strategy="all",
        n_neighbors=3,
        kind_sel="all",
        n_jobs=-1
    )
)

X_train_balanced, y_train_balanced = (
    edited_nearest_neighbours.fit_resample(
        X_train_smote,
        y_train_smote
    )
)

print("Distribution after ENN:")
print(
    pd.Series(y_train_balanced)
    .value_counts()
    .sort_index()
)

print(
    "Balanced training shape:",
    X_train_balanced.shape
)

Distribution after ENN:
0    78415
1    79395
Name: count, dtype: int64
Balanced training shape: (157810, 70)


In [ ]:
number_of_features = X_train_balanced.shape[1]

X_train_transformer = (
    X_train_balanced
    .reshape(
        X_train_balanced.shape[0],
        number_of_features,
        1
    )
    .astype(np.float32)
)

X_test_transformer = (
    X_test_scaled
    .reshape(
        X_test_scaled.shape[0],
        number_of_features,
        1
    )
    .astype(np.float32)
)

y_train_balanced_array = np.asarray(
    y_train_balanced,
    dtype=np.float32
)

y_test_float_array = np.asarray(
    y_test_array,
    dtype=np.float32
)

print(
    "Transformer training shape:",
    X_train_transformer.shape
)

print(
    "Transformer test shape:",
    X_test_transformer.shape
)

Transformer training shape: (157810, 70, 1)
Transformer test shape: (24990, 70, 1)


In [ ]:
transformer_input = tf.keras.Input(
    shape=(number_of_features, 1),
    name="transformer_input"
)

attention_output = (
    tf.keras.layers.MultiHeadAttention(
        num_heads=8,
        key_dim=64,
        output_shape=1,
        name="multi_head_attention"
    )(
        transformer_input,
        transformer_input
    )
)

attention_output = (
    tf.keras.layers.LayerNormalization(
        epsilon=1e-6,
        name="attention_layer_normalization"
    )(
        attention_output
    )
)

attention_residual = tf.keras.layers.Add(
    name="attention_residual_connection"
)([
    transformer_input,
    attention_output
])

feed_forward_output = tf.keras.layers.Dense(
    units=512,
    activation="relu",
    name="feed_forward_dense_512_relu"
)(
    attention_residual
)

feed_forward_output = tf.keras.layers.Dense(
    units=512,
    activation=None,
    name="feed_forward_dense_512_linear"
)(
    feed_forward_output
)

projected_attention_residual = (
    tf.keras.layers.Dense(
        units=512,
        activation=None,
        name="residual_projection_512"
    )(
        attention_residual
    )
)

transformer_output = tf.keras.layers.Add(
    name="feed_forward_residual_connection"
)([
    projected_attention_residual,
    feed_forward_output
])

transformer_output = (
    tf.keras.layers.LayerNormalization(
        epsilon=1e-6,
        name="feed_forward_layer_normalization"
    )(
        transformer_output
    )
)

transformer_output = tf.keras.layers.Flatten(
    name="transformer_flatten"
)(
    transformer_output
)

dnn_output = tf.keras.layers.Dense(
    units=256,
    activation="relu",
    name="dnn_dense_256"
)(
    transformer_output
)

dnn_output = tf.keras.layers.Dropout(
    rate=0.0000001,
    name="dnn_dropout_1"
)(
    dnn_output
)

dnn_output = tf.keras.layers.Dense(
    units=128,
    activation="relu",
    name="dnn_dense_128"
)(
    dnn_output
)

dnn_output = tf.keras.layers.Dropout(
    rate=0.0000001,
    name="dnn_dropout_2"
)(
    dnn_output
)

dnn_output = tf.keras.layers.Dense(
    units=64,
    activation="relu",
    name="dnn_dense_64"
)(
    dnn_output
)

binary_output = tf.keras.layers.Dense(
    units=1,
    activation="sigmoid",
    name="binary_output"
)(
    dnn_output
)

transformer_dnn_model = tf.keras.Model(
    inputs=transformer_input,
    outputs=binary_output,
    name="transformer_dnn_cicids2017_binary"
)

transformer_dnn_model.summary()

Model: "transformer_dnn_cicids2017_binary"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ transformer_input   │ (None, 70, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 70, 1)     │      3,585 │ transformer_inpu… │
│ (MultiHeadAttentio… │                   │            │ transformer_inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer_no… │ (None, 70, 1)     │          2 │ multi_head_atten… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_residual… │ (None, 70, 1)     │          0 │ transformer_inpu… │
│ (Add)               │                   │            │ attention_layer_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feed_forward_dense… │ (None, 70, 512)   │      1,024 │ attention_residu… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_projectio… │ (None, 70, 512)   │      1,024 │ attention_residu… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feed_forward_dense… │ (None, 70, 512)   │    262,656 │ feed_forward_den… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feed_forward_resid… │ (None, 70, 512)   │          0 │ residual_project… │
│ (Add)               │                   │            │ feed_forward_den… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feed_forward_layer… │ (None, 70, 512)   │      1,024 │ feed_forward_res… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_flatten │ (None, 35840)     │          0 │ feed_forward_lay… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnn_dense_256       │ (None, 256)       │  9,175,296 │ transformer_flat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnn_dropout_1       │ (None, 256)       │          0 │ dnn_dense_256[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnn_dense_128       │ (None, 128)       │     32,896 │ dnn_dropout_1[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnn_dropout_2       │ (None, 128)       │          0 │ dnn_dense_128[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dnn_dense_64        │ (None, 64)        │      8,256 │ dnn_dropout_2[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ binary_output       │ (None, 1)         │         65 │ dnn_dense_64[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,485,828 (36.19 MB)

 Trainable params: 9,485,828 (36.19 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
transformer_dnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),
        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5
        ),
        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5
        ),
        tf.keras.metrics.AUC(
            name="roc_auc",
            curve="ROC"
        ),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

In [ ]:
training_history = transformer_dnn_model.fit(
    X_train_transformer,
    y_train_balanced_array,
    validation_split=0.10,
    epochs=500,
    batch_size=128,
    shuffle=True,
    verbose=1
)

Epoch 1/500
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 53s 29ms/step - accuracy: 0.9601 - loss: 0.1262 - pr_auc: 0.9891 - precision: 0.9560 - recall: 0.9547 - roc_auc: 0.9904 - val_accuracy: 0.8084 - val_loss: 0.6004 - val_pr_auc: 1.0000 - val_precision: 1.0000 - val_recall: 0.8084 - val_roc_auc: 0.0000e+00
Epoch 2/500
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9868 - loss: 0.0381 - pr_auc: 0.9978 - precision: 0.9855 - recall: 0.9850 - roc_auc: 0.9985 - val_accuracy: 0.8914 - val_loss: 0.4008 - val_pr_auc: 1.0000 - val_precision: 1.0000 - val_recall: 0.8914 - val_roc_auc: 0.0000e+00
Epoch 3/500
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9887 - loss: 0.0323 - pr_auc: 0.9982 - precision: 0.9873 - recall: 0.9875 - roc_auc: 0.9988 - val_accuracy: 0.9098 - val_loss: 0.1994 - val_pr_auc: 1.0000 - val_precision: 1.0000 - val_recall: 0.9098 - val_roc_auc: 0.0000e+00
Epoch 4/500
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9846 - loss: 0.0460 - pr_auc: 0.9968 - preci

In [ ]:
test_probabilities = (
    transformer_dnn_model.predict(
        X_test_transformer,
        batch_size=128,
        verbose=1
    )
    .reshape(-1)
)

decision_threshold = 0.5

y_pred = (
    test_probabilities >= decision_threshold
).astype(np.int8)

print(
    "Predicted benign:",
    int((y_pred == 0).sum())
)

print(
    "Predicted attack:",
    int((y_pred == 1).sum())
)

196/196 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step
Predicted benign: 13990
Predicted attack: 11000


In [ ]:
cm = confusion_matrix(y_test_array, y_pred)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_array, y_pred)

recall = recall_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[13982    53]
 [    8 10947]]

Attack-class metrics
--------------------
Accuracy : 0.9976 (99.76%)
Detection rate : 0.9993 (99.93%)
Precision      : 0.9952 (99.52%)
FPR            : 0.0038 (0.38%)
F1 score       : 0.9972 (99.72%)


In [ ]:
transformer_dnn_model.save("/content/drive/MyDrive/Models_Table_7/Transformer_DNN_Kamal.keras")

# **Z-K-R framework on CIC-IDS2017 binary classification**

In [ ]:
random.seed(0)
np.random.seed(0)

In [ ]:
df.columns = (
    df.columns
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

print("Number of columns:", len(df.columns))
print(df.columns.tolist())

Number of columns: 79
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag C

In [ ]:
duplicated_column_names = df.columns[
    df.columns.duplicated(keep=False)
].tolist()

print("Duplicated column names:")
print(duplicated_column_names)

df = df.loc[
    :,
    ~df.columns.duplicated(keep="first")
].copy()

print("Shape after duplicate-column removal:", df.shape)

Duplicated column names:
[]
Shape after duplicate-column removal: (2830743, 79)


In [ ]:
label_candidates = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
]

print("Label candidates:", label_candidates)

if len(label_candidates) != 1:
    raise ValueError(
        "Exactly one label column was expected."
    )

label_column = label_candidates[0]

df[label_column] = (
    df[label_column]
    .astype(str)
    .str.strip()
)

print(df[label_column].value_counts(dropna=False))

Label candidates: ['Label']
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [ ]:
label_column = [
    column
    for column in df.columns
    if column.lower() in {
        "label",
        "class",
        "target"
    }
][0]

df["binary_target"] = np.where(
    df[label_column].str.upper().eq("BENIGN"),
    0,
    1
).astype(np.int8)

binary_summary = pd.DataFrame({
    "count": (
        df["binary_target"]
        .value_counts()
        .sort_index()
    ),
    "proportion": (
        df["binary_target"]
        .value_counts(normalize=True)
        .sort_index()
    )
})

binary_summary.index = [
    "BENIGN",
    "ATTACK"
]

binary_summary

,count,proportion
BENIGN,2273097,0.803004
ATTACK,557646,0.196996


In [ ]:
identifier_names = {
    "flow id",
    "flowid",
    "source ip",
    "src ip",
    "destination ip",
    "dst ip",
    "timestamp",
    "source_file"
}

identifier_columns = [
    column
    for column in df.columns
    if column.lower() in identifier_names
]

print("Identifier columns removed:")
print(identifier_columns)

X_raw = df.drop(
    columns=(
        identifier_columns
        + [label_column, "binary_target"]
    ),
    errors="ignore"
).copy()

y_raw = df["binary_target"].copy()

print("Raw predictor shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

Identifier columns removed:
[]
Raw predictor shape: (2830743, 78)
Target shape: (2830743,)


In [ ]:
for column in X_raw.columns:
    X_raw[column] = pd.to_numeric(
        X_raw[column],
        errors="coerce"
    )

X_raw = X_raw.replace(
    [np.inf, -np.inf],
    np.nan
)

invalid_counts = (
    X_raw
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Columns containing invalid values:")
print(invalid_counts[invalid_counts > 0])

Columns containing invalid values:
Flow Packets/s    2867
Flow Bytes/s      2867
dtype: int64


In [ ]:
complete_row_mask = X_raw.notna().all(axis=1)

X_complete = (
    X_raw.loc[complete_row_mask]
    .reset_index(drop=True)
)

y_complete = (
    y_raw.loc[complete_row_mask]
    .reset_index(drop=True)
)

print(
    "Rows removed because of NaN or infinity:",
    int((~complete_row_mask).sum())
)

print("Clean predictor shape:", X_complete.shape)
print("Clean target shape:", y_complete.shape)

Rows removed because of NaN or infinity: 2867
Clean predictor shape: (2827876, 78)
Clean target shape: (2827876,)


In [ ]:
constant_columns = [
    column
    for column in X_complete.columns
    if X_complete[column].nunique(
        dropna=False
    ) <= 1
]

print("Constant predictors:")
print(constant_columns)

X_complete = X_complete.drop(
    columns=constant_columns
)

print("Remaining predictors:", X_complete.shape[1])

Constant predictors:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Remaining predictors: 70


In [ ]:
constant_columns = [
    column
    for column in X_complete.columns
    if X_complete[column].nunique(
        dropna=False
    ) <= 1
]

print("Constant predictors:")
print(constant_columns)

X_complete = X_complete.drop(
    columns=constant_columns
)

print("Remaining predictors:", X_complete.shape[1])

Constant predictors:
[]
Remaining predictors: 70


In [ ]:
X_train_frame, X_test_frame, y_train, y_test = (
    train_test_split(
        X_complete,
        y_complete,
        test_size=0.20,
        random_state=0,
        shuffle=True,
        stratify=y_complete
    )
)

print("Training shape:", X_train_frame.shape)
print("Testing shape:", X_test_frame.shape)

print("\nTraining proportions:")
print(
    y_train
    .value_counts(normalize=True)
    .sort_index()
)

print("\nTesting proportions:")
print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
)

Training shape: (2262300, 70)
Testing shape: (565576, 70)

Training proportions:
binary_target
0    0.803189
1    0.196811
Name: proportion, dtype: float64

Testing proportions:
binary_target
0    0.80319
1    0.19681
Name: proportion, dtype: float64


In [ ]:
zscore_scaler = StandardScaler()

X_train_zscores = zscore_scaler.fit_transform(
    X_train_frame
)

X_test_zscores = zscore_scaler.transform(
    X_test_frame
)

X_train_zscores = X_train_zscores.astype(
    np.float32
)

X_test_zscores = X_test_zscores.astype(
    np.float32
)

print(
    "Largest absolute training Z-score:",
    float(np.abs(X_train_zscores).max())
)

print(
    "Largest absolute test Z-score:",
    float(np.abs(X_test_zscores).max())
)

Largest absolute training Z-score: 1206.5927734375
Largest absolute test Z-score: 3942.953369140625


In [ ]:
training_outlier_mask = (
    np.abs(X_train_zscores) > 3
).any(axis=1)

print(
    "Training outliers identified:",
    int(training_outlier_mask.sum())
)

print(
    "Training outlier percentage:",
    float(training_outlier_mask.mean() * 100)
)

X_train_inlier_zscores = X_train_zscores[
    ~training_outlier_mask
]

y_train_inliers = (
    y_train
    .iloc[
        np.flatnonzero(
            ~training_outlier_mask
        )
    ]
    .reset_index(drop=True)
)

print(
    "Training shape after outlier removal:",
    X_train_inlier_zscores.shape
)

print(
    y_train_inliers
    .value_counts()
    .sort_index()
)

Training outliers identified: 884201
Training outlier percentage: 39.08416213587941
Training shape after outlier removal: (1378099, 70)
binary_target
0    1159755
1     218344
Name: count, dtype: int64


In [ ]:
test_outlier_mask = (
    np.abs(X_test_zscores) > 3
).any(axis=1)

print(
    "Test observations exceeding the training Z-score threshold:",
    int(test_outlier_mask.sum())
)

print(
    "Test rows removed:",
    0
)

Test observations exceeding the training Z-score threshold: 222027
Test rows removed: 0


In [ ]:
from sklearn.decomposition import PCA
pca_model = PCA(
    n_components=10,
    random_state=0
)

X_train_pca = pca_model.fit_transform(
    X_train_inlier_zscores
)

X_test_pca = pca_model.transform(
    X_test_zscores
)

X_train_pca = X_train_pca.astype(
    np.float32
)

X_test_pca = X_test_pca.astype(
    np.float32
)

print("PCA training shape:", X_train_pca.shape)
print("PCA test shape:", X_test_pca.shape)

PCA training shape: (1378099, 10)
PCA test shape: (565576, 10)


In [ ]:
explained_variance_table = pd.DataFrame({
    "component": [
        f"PC{component_number}"
        for component_number in range(1, 11)
    ],
    "explained_variance_ratio": (
        pca_model.explained_variance_ratio_
    )
})

explained_variance_table[
    "cumulative_explained_variance"
] = explained_variance_table[
    "explained_variance_ratio"
].cumsum()

explained_variance_table

,component,explained_variance_ratio,cumulative_explained_variance
0,PC1,0.356903,0.356903
1,PC2,0.231846,0.588749
2,PC3,0.101084,0.689834
3,PC4,0.063401,0.753234
4,PC5,0.049193,0.802427
5,PC6,0.041899,0.844326
6,PC7,0.031436,0.875762
7,PC8,0.025314,0.901075
8,PC9,0.024017,0.925093
9,PC10,0.018066,0.943158


In [ ]:
from sklearn.cluster import KMeans
kmeans_model = KMeans(
    n_clusters=2,
    random_state=0
)

training_cluster_ids = kmeans_model.fit_predict(
    X_train_pca
)

test_cluster_ids = kmeans_model.predict(
    X_test_pca
)

print("Training cluster counts:")
print(
    pd.Series(training_cluster_ids)
    .value_counts()
    .sort_index()
)

print("\nTest cluster counts:")
print(
    pd.Series(test_cluster_ids)
    .value_counts()
    .sort_index()
)

Training cluster counts:
0    395717
1    982382
Name: count, dtype: int64

Test cluster counts:
0    216213
1    349363
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score
)
training_silhouette_score = silhouette_score(
    X_train_pca,
    training_cluster_ids,
    sample_size=min(
        100000,
        len(X_train_pca)
    ),
    random_state=0
)

training_davies_bouldin_score = (
    davies_bouldin_score(
        X_train_pca,
        training_cluster_ids
    )
)

print(
    "Training silhouette score:",
    training_silhouette_score
)

print(
    "Training Davies-Bouldin index:",
    training_davies_bouldin_score
)

Training silhouette score: 0.42873693
Training Davies-Bouldin index: 1.2591071392131836


In [ ]:
cluster_label_table = pd.crosstab(
    pd.Series(
        training_cluster_ids,
        name="cluster"
    ),
    pd.Series(
        y_train_inliers.to_numpy(),
        name="binary_target"
    ),
    normalize="index"
)

cluster_label_table.columns = [
    "BENIGN proportion",
    "ATTACK proportion"
]

cluster_label_table

,BENIGN proportion,ATTACK proportion
cluster,,
0,0.649057,0.350943
1,0.919105,0.080895


In [ ]:
X_train_zkr = np.column_stack([
    X_train_pca,
    training_cluster_ids
]).astype(np.float32)

X_test_zkr = np.column_stack([
    X_test_pca,
    test_cluster_ids
]).astype(np.float32)

zkr_feature_names = [
    "PC1",
    "PC2",
    "PC3",
    "PC4",
    "PC5",
    "PC6",
    "PC7",
    "PC8",
    "PC9",
    "PC10",
    "cluster_id"
]

print("Z-K-R training shape:", X_train_zkr.shape)
print("Z-K-R testing shape:", X_test_zkr.shape)

Z-K-R training shape: (1378099, 11)
Z-K-R testing shape: (565576, 11)


In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=0
)

print(random_forest_model)

RandomForestClassifier(random_state=0)


In [ ]:
random_forest_model.fit(
    X_train_zkr,
    y_train_inliers
)



RandomForestClassifier(random_state=0)

In [ ]:
y_pred = random_forest_model.predict(
    X_test_zkr
)

cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test, y_pred)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[452407   1858]
 [ 23899  87412]]

Attack-class metrics
--------------------
Accuracy : 0.9545 (95.45%)
Detection rate : 0.7853 (78.53%)
Precision      : 0.9792 (97.92%)
FPR            : 0.0041 (0.41%)
F1 score       : 0.8716 (87.16%)


# **Evolving Deep Learning Architectures for Network Intrusion Detection Using a Double PSO Metaheuristic.**

In [5]:
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

In [25]:
df.columns = (
    df.columns
    .astype(str)
    .str.replace(
        "\ufeff",
        "",
        regex=False
    )
    .str.strip()
)

print("Number of loaded columns:", len(df.columns))
print(df.columns.tolist())

Number of loaded columns: 79
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE

In [26]:
label_positions = [
    position
    for position, column in enumerate(df.columns)
    if column.lower() in {
        "label",
        "class",
        "target"
    }
]

print("Possible label positions:", label_positions)

if len(label_positions) != 1:
    raise ValueError(
        "Exactly one label column was expected."
    )

label_position = label_positions[0]
label_column_name = df.columns[label_position]

print("Label position:", label_position)
print("Label name:", label_column_name)

Possible label positions: [78]
Label position: 78
Label name: Label


In [27]:
label_values = (
    df.iloc[:, label_position]
    .astype(str)
    .str.strip()
)

binary_target = np.where(
    label_values.str.upper().eq("BENIGN"),
    0,
    1
).astype(np.int8)

print("Raw binary counts:")
print(
    pd.Series(binary_target)
    .value_counts()
    .sort_index()
)

Raw binary counts:
0    2273097
1     557646
Name: count, dtype: int64


In [31]:
predictor_positions = [
    position
    for position in range(df.shape[1])
    if position != label_position
    and df.columns[position] != "source_file"
]

X_ordered = df.iloc[
    :,
    predictor_positions
].copy()

original_predictor_names = [
    df.columns[position]
    for position in predictor_positions
]

X_ordered.columns = [
    f"feature_{position + 1}"
    for position in range(X_ordered.shape[1])
]

y_ordered = pd.Series(
    binary_target,
    name="binary_target",
    index=X_ordered.index
)

print("Ordered predictor count:", X_ordered.shape[1])
print("Target count:", len(y_ordered))

Ordered predictor count: 78
Target count: 2830743


In [32]:
for column in X_ordered.columns:
    X_ordered[column] = pd.to_numeric(
        X_ordered[column],
        errors="coerce"
    )

X_ordered = X_ordered.replace(
    [np.inf, -np.inf],
    np.nan
)

invalid_counts = (
    X_ordered
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Predictors containing invalid values:")
print(invalid_counts[invalid_counts > 0])

Predictors containing invalid values:
feature_16    2867
feature_15    2867
dtype: int64


In [33]:
complete_row_mask = X_ordered.notna().all(axis=1)

X_complete = (
    X_ordered.loc[complete_row_mask]
    .reset_index(drop=True)
)

y_complete = (
    y_ordered.loc[complete_row_mask]
    .reset_index(drop=True)
)

print(
    "Rows removed:",
    int((~complete_row_mask).sum())
)

print("Complete predictor shape:", X_complete.shape)
print("Complete target shape:", y_complete.shape)

Rows removed: 2867
Complete predictor shape: (2827876, 78)
Complete target shape: (2827876,)


In [34]:
paper_selected_one_based_indices = [
    8,
    11,
    15,
    18,
    20,
    23,
    24,
    26,
    28,
    31,
    33,
    37,
    43,
    44,
    51,
    53,
    54,
    59,
    70,
    73,
    74,
    77]

paper_selected_zero_based_indices = [
    index - 1
    for index in paper_selected_one_based_indices
]

print(
    "Selected feature count:",
    len(paper_selected_zero_based_indices)
)

Selected feature count: 22


In [35]:
selected_feature_columns = [
    X_complete.columns[index]
    for index in paper_selected_zero_based_indices
]

selected_original_names = [
    original_predictor_names[index]
    for index in paper_selected_zero_based_indices
]

selected_feature_table = pd.DataFrame({
    "paper_position": paper_selected_one_based_indices,
    "internal_column": selected_feature_columns,
    "original_name": selected_original_names
})

selected_feature_table

,paper_position,internal_column,original_name
0,8,feature_8,Fwd Packet Length Min
1,11,feature_11,Bwd Packet Length Max
2,15,feature_15,Flow Bytes/s
3,18,feature_18,Flow IAT Std
4,20,feature_20,Flow IAT Min
5,23,feature_23,Fwd IAT Std
6,24,feature_24,Fwd IAT Max
7,26,feature_26,Bwd IAT Total
8,28,feature_28,Bwd IAT Std
9,31,feature_31,Fwd PSH Flags


In [36]:
X_selected = X_complete[
    selected_feature_columns
].copy()

y_selected = y_complete.copy()

print("Selected predictor shape:", X_selected.shape)
print("Target shape:", y_selected.shape)

Selected predictor shape: (2827876, 22)
Target shape: (2827876,)


In [37]:
benign_indices = y_selected[
    y_selected == 0
].index.to_numpy()

attack_indices = y_selected[
    y_selected == 1
].index.to_numpy()

print("Available benign records:", len(benign_indices))
print("Available attack records:", len(attack_indices))

Available benign records: 2271320
Available attack records: 556556


In [38]:
random_generator = np.random.default_rng(0)

shuffled_benign_indices = random_generator.permutation(
    benign_indices
)

shuffled_attack_indices = random_generator.permutation(
    attack_indices
)

training_benign_indices = shuffled_benign_indices[
    :18750
]

testing_benign_indices = shuffled_benign_indices[
    18750:37500
]

training_attack_indices = shuffled_attack_indices[
    :131250
]

testing_attack_indices = shuffled_attack_indices[
    131250:262500
]

training_indices = np.concatenate([
    training_benign_indices,
    training_attack_indices
])

testing_indices = np.concatenate([
    testing_benign_indices,
    testing_attack_indices
])

random_generator.shuffle(training_indices)
random_generator.shuffle(testing_indices)

print("Training records:", len(training_indices))
print("Testing records:", len(testing_indices))
print(
    "Overlapping records:",
    len(
        np.intersect1d(
            training_indices,
            testing_indices
        )
    )
)

Training records: 150000
Testing records: 150000
Overlapping records: 0


In [40]:
X_train_frame = (
    X_selected
    .loc[training_indices]
    .reset_index(drop=True)
)

y_train = (
    y_selected
    .loc[training_indices]
    .reset_index(drop=True)
)

X_test_frame = (
    X_selected
    .loc[testing_indices]
    .reset_index(drop=True)
)

y_test = (
    y_selected
    .loc[testing_indices]
    .reset_index(drop=True)
)

print("Training shape:", X_train_frame.shape)
print("Testing shape:", X_test_frame.shape)

print("\nTraining class counts:")
print(y_train.value_counts().sort_index())

print("\nTesting class counts:")
print(y_test.value_counts().sort_index())

Training shape: (150000, 22)
Testing shape: (150000, 22)

Training class counts:
binary_target
0     18750
1    131250
Name: count, dtype: int64

Testing class counts:
binary_target
0     18750
1    131250
Name: count, dtype: int64


In [41]:
minmax_scaler = MinMaxScaler(
    feature_range=(0, 1)
)

X_train_scaled = minmax_scaler.fit_transform(
    X_train_frame
)

X_test_scaled = minmax_scaler.transform(
    X_test_frame
)

X_train_scaled = X_train_scaled.astype(
    np.float32
)

X_test_scaled = X_test_scaled.astype(
    np.float32
)

y_train_array = y_train.to_numpy(
    dtype=np.int32
)

y_test_array = y_test.to_numpy(
    dtype=np.int32
)

print("Training minimum:", X_train_scaled.min())
print("Training maximum:", X_train_scaled.max())

Training minimum: 0.0
Training maximum: 1.0


In [42]:
learning_rate_schedule = (
    tf.keras.optimizers.schedules.InverseTimeDecay(
        initial_learning_rate=0.1,
        decay_steps=1,
        decay_rate=0.005,
        staircase=False
    )
)

print(
    "Initial learning rate:",
    float(learning_rate_schedule(0))
)

print(
    "Learning rate after 100 updates:",
    float(learning_rate_schedule(100))
)

Initial learning rate: 0.10000000149011612
Learning rate after 100 updates: 0.06666667014360428


In [46]:
tf.keras.backend.clear_session()
tf.random.set_seed(0)

dnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(22,)
    ),

    tf.keras.layers.Dense(
        units=23,
        activation="tanh",
        kernel_initializer=tf.keras.initializers.RandomUniform(
            minval=-0.05,
            maxval=0.05,
            seed=0
        ),
        name="dense_input_23"
    ),

    tf.keras.layers.Dropout(
        rate=0.25,
        seed=0,
        name="dropout_input"
    ),

    tf.keras.layers.Dense(
        units=15,
        activation="softsign",
        kernel_initializer=tf.keras.initializers.RandomUniform(
            minval=-0.05,
            maxval=0.05,
            seed=1
        ),
        name="dense_hidden_15"
    ),

    tf.keras.layers.Dropout(
        rate=0.25,
        seed=1,
        name="dropout_hidden_1"
    ),

    tf.keras.layers.Dense(
        units=6,
        activation="softplus",
        kernel_initializer=tf.keras.initializers.RandomUniform(
            minval=-0.05,
            maxval=0.05,
            seed=2
        ),
        name="dense_hidden_6"
    ),

    tf.keras.layers.Dropout(
        rate=0.25,
        seed=2,
        name="dropout_hidden_2"
    ),

    tf.keras.layers.Dense(
        units=2,
        activation="softmax",
        kernel_initializer=tf.keras.initializers.RandomUniform(
            minval=-0.05,
            maxval=0.05,
            seed=3
        ),
        name="binary_softmax_output"
    )
])

dnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_input_23 (Dense)          │ (None, 23)             │           529 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_input (Dropout)         │ (None, 23)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden_15 (Dense)         │ (None, 15)             │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_hidden_1 (Dropout)      │ (None, 15)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden_6 (Dense)          │ (None, 6)              │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_hidden_2 (Dropout)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ binary_softmax_output (Dense)   │ (None, 2)              │            14 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 999 (3.90 KB)

 Trainable params: 999 (3.90 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
dnn_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=learning_rate_schedule,
        momentum=0.3
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

In [48]:
training_history = dnn_model.fit(
    X_train_scaled,
    y_train_array,
    epochs=15,
    batch_size=450,
    shuffle=True,
    verbose=1
)

Epoch 1/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.8810 - loss: 0.3025
Epoch 2/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9289 - loss: 0.2230
Epoch 3/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9354 - loss: 0.2090
Epoch 4/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9407 - loss: 0.1978
Epoch 5/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9427 - loss: 0.1939
Epoch 6/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9440 - loss: 0.1899
Epoch 7/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9452 - loss: 0.1870
Epoch 8/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9456 - loss: 0.1858
Epoch 9/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9458 - loss: 0.1850
Epoch 10/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9467 - loss: 0.1832
Epoch 11/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9466 - loss: 0.1820
Epoch 12/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/st

In [50]:
test_class_probabilities = dnn_model.predict(
    X_test_scaled,
    batch_size=450,
    verbose=1
)

test_attack_probabilities = (
    test_class_probabilities[:, 1]
)

y_pred = np.argmax(
    test_class_probabilities,
    axis=1
).astype(np.int8)

print(
    "Predicted normal:",
    int((y_pred == 0).sum())
)

print(
    "Predicted attack:",
    int((y_pred == 1).sum())
)

334/334 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Predicted normal: 15296
Predicted attack: 134704


In [51]:

cm = confusion_matrix(y_test_array, y_pred)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_array, y_pred)

recall = recall_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[ 13305   5445]
 [  1991 129259]]

Attack-class metrics
--------------------
Accuracy : 0.9504 (95.04%)
Detection rate : 0.9848 (98.48%)
Precision      : 0.9596 (95.96%)
FPR            : 0.2904 (29.04%)
F1 score       : 0.9720 (97.20%)


In [52]:
dnn_model.save("/content/drive/MyDrive/Models_Table_7/DNN_Elmasry.keras")